Connect to our Bronze tables

In [0]:
catalog = spark.sql(
    "SELECT current_catalog()"
).first()[0]

schema_name = "shopsphere"

print("Catalog:", catalog)
print("Schema:", schema_name)

Load Bronze datasets

In [0]:
bronze_tables = {
    "customers": f"{catalog}.{schema_name}.bronze_customers",
    "products": f"{catalog}.{schema_name}.bronze_products",
    "orders": f"{catalog}.{schema_name}.bronze_orders",
    "order_items": f"{catalog}.{schema_name}.bronze_order_items",
    "payments": f"{catalog}.{schema_name}.bronze_payments"
}

bronze_dataframes = {}

for name, table_name in bronze_tables.items():

    df = spark.table(table_name)

    bronze_dataframes[name] = df

    print(f"{name}: {df.count()} records")

Inspect our customer data

In [0]:
customers_df = bronze_dataframes["customers"]

customers_df.show(10, truncate=False)

customers_df.printSchema()

In [0]:
from pyspark.sql.functions import col, count, when

customers_df.select(
    count(
        when(col("customer_name").isNull(), 1)
    ).alias("missing_names"),

    count(
        when(col("city").isNull(), 1)
    ).alias("missing_cities")
).show()

In [0]:
print("Total records:", customers_df.count())

print(
    "Unique customer IDs:",
    customers_df.select("customer_id").distinct().count()
)

In [0]:
customers_silver = customers_df.dropDuplicates(["customer_id"])

Handle missing values

In [0]:
customers_silver.select(
    count(when(col("customer_name").isNull(), 1)).alias("missing_names"),
    count(when(col("city").isNull(), 1)).alias("missing_cities"),
    count(when(col("email").isNull(), 1)).alias("missing_emails")
).show()

Customer cleaning and quarantine

In [0]:
from pyspark.sql.functions import col, when, lit, trim, lower

customers_prepared = customers_silver.withColumn(
    "customer_name", trim(col("customer_name"))
).withColumn(
    "city", trim(col("city"))
).withColumn(
    "email", lower(trim(col("email")))
)

# Separate invalid records
customer_quarantine = customers_prepared.filter(
    col("customer_id").isNull() |
    (col("customer_id") == "") |
    col("customer_name").isNull() |
    (col("customer_name") == "")
)

# Keep valid records
customers_clean = customers_prepared.filter(
    col("customer_id").isNotNull() &
    (col("customer_id") != "") &
    col("customer_name").isNotNull() &
    (col("customer_name") != "")
).withColumn(
    "city",
    when(
        col("city").isNull() | (col("city") == ""),
        lit("Unknown")
    ).otherwise(col("city"))
)

print("Clean customer records:", customers_clean.count())
print("Quarantined records:", customer_quarantine.count())

Verify the results

In [0]:
customers_clean.select(
    count(when(col("customer_name").isNull(), 1)).alias("missing_names"),
    count(when(col("city") == "Unknown", 1)).alias("unknown_cities"),
    count(when(col("email").isNull(), 1)).alias("missing_emails")
).show()

customer_quarantine.show(10, truncate=False)

In [0]:
table_configs = {
    "products": {
        "key": "product_id",
        "required": ["product_id", "product_name", "category"],
        "numeric": ["unit_price"]
    },

    "orders": {
        "key": "order_id",
        "required": ["order_id", "customer_id", "order_date"],
        "numeric": []
    },

    "order_items": {
        "key": "item_id",
        "required": ["item_id", "order_id", "product_id"],
        "numeric": ["quantity", "unit_price"]
    },

    "payments": {
        "key": "payment_id",
        "required": ["payment_id", "order_id", "payment_date", "amount"],
        "numeric": ["amount"]
    }
}

Create the reusable cleaning function

In [0]:

from pyspark.sql.functions import col, trim, when, lit, to_date, expr

def clean_dataset(df, config):

    key = config["key"]
    required = config["required"]
    numeric_columns = config["numeric"]

    # 1. Trim string columns
    for column_name, data_type in df.dtypes:
        if data_type == "string":
            df = df.withColumn(
                column_name,
                trim(col(column_name))
            )

    # 2. Remove exact duplicate rows
    df = df.dropDuplicates()

    # 3. Remove duplicate business keys
    df = df.dropDuplicates([key])

    # 4. Identify missing required fields
    invalid_condition = None

    for column_name in required:
        condition = (
            col(column_name).isNull() |
            (col(column_name) == "")
        )

        invalid_condition = (
            condition if invalid_condition is None
            else invalid_condition | condition
        )

    # 5. Validate numeric columns safely
    for column_name in numeric_columns:

        parsed_column = f"_parsed_{column_name}"

        df = df.withColumn(
            parsed_column,
            expr(f"try_cast(`{column_name}` AS DOUBLE)")
        )

        condition = (
            col(parsed_column).isNull() &
            col(column_name).isNotNull() &
            (trim(col(column_name)) != "")
        )

        invalid_condition = (
            condition if invalid_condition is None
            else invalid_condition | condition
        )

    # 6. Separate clean and rejected records
    quarantine = df.filter(invalid_condition)
    clean = df.filter(~invalid_condition)

    # 7. Convert numeric columns in clean data
    for column_name in numeric_columns:
        clean = clean.withColumn(
            column_name,
            expr(f"try_cast(`{column_name}` AS DOUBLE)")
        )

    # Remove temporary columns from both outputs
    for column_name in numeric_columns:
        quarantine = quarantine.drop(f"_parsed_{column_name}")

    return clean, quarantine


In [0]:
silver_dataframes = {
    "customers": customers_clean
}

quarantine_dataframes = {
    "customers": customer_quarantine
}

for name, config in table_configs.items():

    source_df = bronze_dataframes[name]

    clean_df, rejected_df = clean_dataset(
        source_df,
        config
    )

    silver_dataframes[name] = clean_df
    quarantine_dataframes[name] = rejected_df

    print(f"\n{name}")
    print("Clean records:", clean_df.count())
    print("Quarantined records:", rejected_df.count())

In [0]:
for name, df in silver_dataframes.items():

    print(f"\n--- {name.upper()} ---")

    print("Records:", df.count())
    df.printSchema()
    df.show(5, truncate=False)

Define business rules

In [0]:
business_rules = {
    "customers": {
        "conditions": []
    },

    "products": {
        "conditions": [
            ("unit_price", "positive")
        ]
    },

    "orders": {
        "conditions": []
    },

    "order_items": {
        "conditions": [
            ("quantity", "positive"),
            ("unit_price", "positive")
        ]
    },

    "payments": {
        "conditions": [
            ("amount", "positive")
        ]
    }
}

Upgrade the cleaning function

In [0]:

from pyspark.sql.functions import (
    col, trim, when, lit, expr, to_date
)

def clean_dataset(df, config, rules):

    key = config["key"]
    required = config["required"]
    numeric_columns = config["numeric"]

    # 1. Trim string columns
    for column_name, data_type in df.dtypes:
        if data_type == "string":
            df = df.withColumn(
                column_name,
                trim(col(column_name))
            )

    # 2. Remove exact duplicate rows
    df = df.dropDuplicates()

    # 3. Remove duplicate business keys
    df = df.dropDuplicates([key])

    # 4. Validate required fields
    invalid_condition = None

    for column_name in required:
        condition = (
            col(column_name).isNull() |
            (col(column_name) == "")
        )

        invalid_condition = (
            condition if invalid_condition is None
            else invalid_condition | condition
        )

    # 5. Validate numeric fields
    for column_name in numeric_columns:

        parsed_column = f"_parsed_{column_name}"

        df = df.withColumn(
            parsed_column,
            expr(f"try_cast(`{column_name}` AS DOUBLE)")
        )

        condition = (
            col(parsed_column).isNull() &
            col(column_name).isNotNull() &
            (trim(col(column_name)) != "")
        )

        invalid_condition = (
            condition if invalid_condition is None
            else invalid_condition | condition
        )

    # 6. Apply business rules using parsed numeric values

    for column_name, rule in rules["conditions"]:

        parsed_column = f"_parsed_{column_name}"

        if rule == "positive":
            condition = col(parsed_column) <= 0

            invalid_condition = (
                condition if invalid_condition is None
                else invalid_condition | condition
            )

    # 7. Separate clean and invalid records
    quarantine = df.filter(invalid_condition)
    clean = df.filter(~invalid_condition)

    # 8. Convert numeric columns in clean data
    for column_name in numeric_columns:
        clean = clean.withColumn(
            column_name,
            expr(f"try_cast(`{column_name}` AS DOUBLE)")
        )

    # 9. Remove temporary columns from quarantine
    for column_name in numeric_columns:
        quarantine = quarantine.drop(
            f"_parsed_{column_name}"
        )

    # 10. Add rejection reason
    quarantine = quarantine.withColumn(
        "rejection_reason",
        lit("Failed data quality validation")
    )

    return clean, quarantine


Apply the framework to all datasets

In [0]:
silver_dataframes = {
    "customers": customers_clean
}

quarantine_dataframes = {
    "customers": customer_quarantine
}

for name, config in table_configs.items():

    clean_df, rejected_df = clean_dataset(
        bronze_dataframes[name],
        config,
        business_rules[name]
    )

    silver_dataframes[name] = clean_df
    quarantine_dataframes[name] = rejected_df

    print(f"\n{name.upper()}")
    print("Clean:", clean_df.count())
    print("Quarantined:", rejected_df.count())

Run the validation summary

In [0]:
for name in silver_dataframes:

    clean_df = silver_dataframes[name]
    rejected_df = quarantine_dataframes[name]

    print(f"\n--- {name.upper()} ---")
    print("Clean:", clean_df.count())
    print("Quarantined:", rejected_df.count())
    print("Total:", clean_df.count() + rejected_df.count())

Inspect the rejected records

In [0]:
for name, df in quarantine_dataframes.items():

    print(f"\n--- {name.upper()} QUARANTINE ---")
    df.show(10, truncate=False)

In [0]:
silver_dataframes["customers"].groupBy("city") \
    .count() \
    .orderBy("city") \
    .show(100, truncate=False)

Define date configurations

In [0]:
date_configs = {
    "customers": ["signup_date"],
    "products": [],
    "orders": ["order_date"],
    "order_items": [],
    "payments": ["payment_date"]
}

Create a reusable date validator

What does this do?
For example, if order_date contains:
Input	Result
2025-06-25	Valid date
2025-13-40	Invalid date
NULL	Invalid date
Empty string	Invalid date

In [0]:
from pyspark.sql.functions import to_date, col, trim

def validate_dates(df, date_columns):

    invalid_condition = None

    for column_name in date_columns:

        parsed_column = f"_parsed_{column_name}"

        df = df.withColumn(
            parsed_column,
            to_date(col(column_name), "yyyy-MM-dd")
        )

        condition = (
            col(parsed_column).isNull() |
            col(column_name).isNull() |
            (trim(col(column_name)) == "")
        )

        invalid_condition = (
            condition if invalid_condition is None
            else invalid_condition | condition
        )

    if invalid_condition is None:
        return df, df.limit(0)

    quarantine = df.filter(invalid_condition)
    clean = df.filter(~invalid_condition)

    for column_name in date_columns:
        parsed_column = f"_parsed_{column_name}"

        clean = clean.withColumn(
            column_name,
            col(parsed_column)
        ).drop(parsed_column)

        quarantine = quarantine.drop(parsed_column)

    return clean, quarantine

Apply it to all five datasets

In [0]:
from functools import reduce

for name, date_columns in date_configs.items():

    if not date_columns:
        continue

    clean_df, date_rejected = validate_dates(
        silver_dataframes[name],
        date_columns
    )

    existing_quarantine = quarantine_dataframes[name]

    combined_quarantine = existing_quarantine.unionByName(
        date_rejected,
        allowMissingColumns=True
    )

    silver_dataframes[name] = clean_df
    quarantine_dataframes[name] = combined_quarantine

    print(f"\n{name.upper()}")
    print("Date-valid records:", clean_df.count())
    print("Total quarantined:", combined_quarantine.count())

Verify date types

In [0]:
for name in ["customers", "orders", "payments"]:

    print(f"\n{name.upper()}")
    silver_dataframes[name].printSchema()

This temporary column should not be present in the final Silver table. We'll remove it after confirming the numeric conversion.

In [0]:
silver_dataframes["payments"] = (
    silver_dataframes["payments"].drop("_parsed_amount")
)

Check relationships

In [0]:
# Build reference ID sets from Silver tables

customer_ids = silver_dataframes["customers"].select(
    "customer_id"
).distinct()

order_ids = silver_dataframes["orders"].select(
    "order_id"
).distinct()

product_ids = silver_dataframes["products"].select(
    "product_id"
).distinct()

Validate orders

In [0]:
from pyspark.sql.functions import col

invalid_orders = silver_dataframes["orders"].join(
    customer_ids,
    on="customer_id",
    how="left_anti"
)

valid_orders = silver_dataframes["orders"].join(
    customer_ids,
    on="customer_id",
    how="left_semi"
)

print("Invalid orders:", invalid_orders.count())
print("Valid orders:", valid_orders.count())

Inspect the invalid orders

In [0]:
invalid_orders.select(
    "order_id",
    "customer_id",
    "order_date"
).show(30, truncate=False)

Check whether they reference quarantined customers

In [0]:
invalid_orders.join(
    customer_quarantine.select("customer_id"),
    on="customer_id",
    how="inner"
).count()

Validate order items

An order item must reference:
- An existing order.
- An existing product.

In [0]:
from pyspark.sql.functions import col, lit, when

order_items_df = silver_dataframes["order_items"]

# Find items with invalid order references
invalid_order_ref = order_items_df.join(
    order_ids,
    on="order_id",
    how="left_anti"
)

# Find items with invalid product references
invalid_product_ref = order_items_df.join(
    product_ids,
    on="product_id",
    how="left_anti"
)

# Combine both sets without duplicating records
invalid_items = invalid_order_ref.unionByName(
    invalid_product_ref
).dropDuplicates(["item_id"])

valid_items = order_items_df.join(
    order_ids, on="order_id", how="left_semi"
).join(
    product_ids, on="product_id", how="left_semi"
)

print("Invalid order items:", invalid_items.count())
print("Valid order items:", valid_items.count())

Validate _payments_

Every payment must reference an existing order.

In [0]:
payments_df = silver_dataframes["payments"]

invalid_payments_ref = payments_df.join(
    order_ids,
    on="order_id",
    how="left_anti"
)

valid_payments = payments_df.join(
    order_ids,
    on="order_id",
    how="left_semi"
)

print("Invalid payments:", invalid_payments_ref.count())
print("Valid payments:", valid_payments.count())

Update Silver and quarantine

In [0]:
# Update orders
silver_dataframes["orders"] = valid_orders

quarantine_dataframes["orders"] = (
    quarantine_dataframes["orders"]
    .unionByName(
        invalid_orders.withColumn(
            "rejection_reason",
            lit("CUSTOMER_NOT_FOUND_IN_SILVER")
        ),
        allowMissingColumns=True
    )
)

# Update order items
silver_dataframes["order_items"] = valid_items

quarantine_dataframes["order_items"] = (
    quarantine_dataframes["order_items"]
    .unionByName(
        invalid_items.withColumn(
            "rejection_reason",
            lit("INVALID_ORDER_OR_PRODUCT_REFERENCE")
        ),
        allowMissingColumns=True
    )
)

# Update payments
silver_dataframes["payments"] = valid_payments

quarantine_dataframes["payments"] = (
    quarantine_dataframes["payments"]
    .unionByName(
        invalid_payments_ref.withColumn(
            "rejection_reason",
            lit("ORDER_NOT_FOUND_IN_SILVER")
        ),
        allowMissingColumns=True
    )
)

Final relationship verification

In [0]:
print("Orders referencing missing customers:",
      silver_dataframes["orders"].join(
          customer_ids,
          on="customer_id",
          how="left_anti"
      ).count())

print("Order items referencing missing orders:",
      silver_dataframes["order_items"].join(
          order_ids,
          on="order_id",
          how="left_anti"
      ).count())

print("Order items referencing missing products:",
      silver_dataframes["order_items"].join(
          product_ids,
          on="product_id",
          how="left_anti"
      ).count())

print("Payments referencing missing orders:",
      silver_dataframes["payments"].join(
          order_ids,
          on="order_id",
          how="left_anti"
      ).count())

Refresh the reference IDs

In [0]:
customer_ids = silver_dataframes["customers"].select(
    "customer_id"
).distinct()

order_ids = silver_dataframes["orders"].select(
    "order_id"
).distinct()

product_ids = silver_dataframes["products"].select(
    "product_id"
).distinct()

Revalidate order items and payments

In [0]:
# Recheck order items against final orders and products
items_df = silver_dataframes["order_items"]

invalid_items_final = items_df.join(
    order_ids, "order_id", "left_anti"
).unionByName(
    items_df.join(product_ids, "product_id", "left_anti")
).dropDuplicates(["item_id"])

valid_items_final = items_df.join(
    order_ids, "order_id", "left_semi"
).join(
    product_ids, "product_id", "left_semi"
)

# Recheck payments against final orders
payments_df = silver_dataframes["payments"]

invalid_payments_final = payments_df.join(
    order_ids, "order_id", "left_anti"
)

valid_payments_final = payments_df.join(
    order_ids, "order_id", "left_semi"
)

# Update Silver DataFrames
silver_dataframes["order_items"] = valid_items_final
silver_dataframes["payments"] = valid_payments_final

# Add newly rejected records to quarantine
quarantine_dataframes["order_items"] = (
    quarantine_dataframes["order_items"].unionByName(
        invalid_items_final.withColumn(
            "rejection_reason",
            lit("INVALID_ORDER_OR_PRODUCT_REFERENCE")
        ),
        allowMissingColumns=True
    )
)

quarantine_dataframes["payments"] = (
    quarantine_dataframes["payments"].unionByName(
        invalid_payments_final.withColumn(
            "rejection_reason",
            lit("ORDER_NOT_FOUND_IN_SILVER")
        ),
        allowMissingColumns=True
    )
)

Run final integrity checks

In [0]:
final_checks = {
    "Orders → Customers": silver_dataframes["orders"].join(
        customer_ids, "customer_id", "left_anti"
    ).count(),

    "Order items → Orders": silver_dataframes["order_items"].join(
        order_ids, "order_id", "left_anti"
    ).count(),

    "Order items → Products": silver_dataframes["order_items"].join(
        product_ids, "product_id", "left_anti"
    ).count(),

    "Payments → Orders": silver_dataframes["payments"].join(
        order_ids, "order_id", "left_anti"
    ).count()
}

for check, failures in final_checks.items():
    print(f"{check}: {failures} invalid references")

Review final Silver and quarantine counts

In [0]:
for name in silver_dataframes:

    clean_count = silver_dataframes[name].count()
    rejected_count = quarantine_dataframes[name].count()

    print(f"\n{name.upper()}")
    print("Silver:", clean_count)
    print("Quarantine:", rejected_count)

Important: audit the quarantine records

In [0]:
quarantine_dataframes["order_items"].groupBy(
    "rejection_reason"
).count().show(truncate=False)

quarantine_dataframes["payments"].groupBy(
    "rejection_reason"
).count().show(truncate=False)

In [0]:
quarantine_dataframes["order_items"].show(
    20, truncate=False
)

quarantine_dataframes["payments"].show(
    20, truncate=False
)

In [0]:
items_df = bronze_dataframes["order_items"]

# Items referencing orders missing from Silver
missing_order_items = items_df.join(
    silver_dataframes["orders"].select("order_id"),
    "order_id",
    "left_anti"
)

# Items referencing products missing from Silver
missing_product_items = items_df.join(
    silver_dataframes["products"].select("product_id"),
    "product_id",
    "left_anti"
)

print("Missing order references:", missing_order_items.count())
print("Missing product references:", missing_product_items.count())

Assign meaningful rejection reasons

In [0]:
from pyspark.sql.functions import col, when, lit, expr, trim

# CUSTOMERS
quarantine_dataframes["customers"] = (
    quarantine_dataframes["customers"]
    .withColumn(
        "rejection_reason",
        lit("MISSING_CUSTOMER_NAME")
    )
)

# PRODUCTS
quarantine_dataframes["products"] = (
    quarantine_dataframes["products"]
    .withColumn(
        "rejection_reason",
        lit("NON_POSITIVE_PRODUCT_PRICE")
    )
)

# ORDERS
quarantine_dataframes["orders"] = (
    quarantine_dataframes["orders"]
    .withColumn(
        "rejection_reason",
        lit("CUSTOMER_NOT_FOUND")
    )
)

# ORDER ITEMS
items = quarantine_dataframes["order_items"]

quarantine_dataframes["order_items"] = items.withColumn(
    "rejection_reason",
    when(
        col("quantity").cast("double") <= 0,
        "INVALID_QUANTITY"
    ).when(
        ~col("order_id").isin(
            *[r.order_id for r in silver_dataframes["orders"].select("order_id").collect()]
        ),
        "ORDER_NOT_FOUND"
    ).otherwise(
        "PRODUCT_NOT_FOUND"
    )
)

# PAYMENTS
payments = quarantine_dataframes["payments"]

quarantine_dataframes["payments"] = payments.withColumn(
    "rejection_reason",
    when(
        expr("try_cast(amount AS DOUBLE)").isNull(),
        "INVALID_PAYMENT_AMOUNT"
    ).otherwise(
        "ORDER_NOT_FOUND"
    )
)

Verify the quarantine summary

In [0]:
for name, df in quarantine_dataframes.items():

    print(f"\n--- {name.upper()} ---")

    df.groupBy("rejection_reason") \
        .count() \
        .show(truncate=False)

Save all five Silver tables

In [0]:
for name, df in silver_dataframes.items():

    table_name = f"{catalog}.{schema_name}.silver_{name}"

    df.write.format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable(table_name)

    print("Saved:", table_name)

Save all five quarantine tables

In [0]:
for name, df in quarantine_dataframes.items():

    table_name = f"{catalog}.{schema_name}.quarantine_{name}"

    df.write.format("delta") \
        .mode("overwrite") \
        .option("overwriteSchema", "true") \
        .saveAsTable(table_name)

    print("Saved:", table_name)

In [0]:
from pyspark.sql.functions import col, expr, lit

# 1. Recover invalid payment amounts from Bronze
invalid_amount_payments = (
    bronze_dataframes["payments"]
    .withColumn(
        "_parsed_amount",
        expr("try_cast(amount AS DOUBLE)")
    )
    .filter(
        col("_parsed_amount").isNull() &
        col("amount").isNotNull()
    )
    .drop("_parsed_amount")
    .dropDuplicates(["payment_id"])
    .withColumn(
        "rejection_reason",
        lit("INVALID_PAYMENT_AMOUNT")
    )
)

# 2. Find valid payments referencing quarantined orders
missing_order_payments = (
    silver_dataframes["payments"]
    .join(
        silver_dataframes["orders"].select("order_id"),
        on="order_id",
        how="left_anti"
    )
    .withColumn(
        "amount",
        col("amount").cast("string")
    )
    .withColumn(
        "rejection_reason",
        lit("ORDER_NOT_FOUND")
    )
)

# 3. Make sure both amount columns have the same type
invalid_amount_payments = invalid_amount_payments.withColumn(
    "amount",
    col("amount").cast("string")
)

# 4. Combine safely
quarantine_dataframes["payments"] = (
    invalid_amount_payments.unionByName(
        missing_order_payments,
        allowMissingColumns=True
    )
)

print(
    "Payment quarantine:",
    quarantine_dataframes["payments"].count()
)

In [0]:
quarantine_dataframes["payments"].write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(
        f"{catalog}.{schema_name}.quarantine_payments"
    )

print("Payment quarantine saved successfully!")

Final verification

In [0]:
for name in silver_dataframes:

    silver_table = f"{catalog}.{schema_name}.silver_{name}"
    quarantine_table = f"{catalog}.{schema_name}.quarantine_{name}"

    print(f"\n{name.upper()}")
    print("Silver:", spark.table(silver_table).count())
    print("Quarantine:", spark.table(quarantine_table).count())

In [0]:
spark.table(
    f"{catalog}.{schema_name}.quarantine_payments"
).groupBy("rejection_reason").count().show()

In [0]:
missing_order_payments = (
    bronze_dataframes["payments"]
    .dropDuplicates(["payment_id"])
    .join(
        silver_dataframes["orders"].select("order_id"),
        on="order_id",
        how="left_anti"
    )
    .filter(expr("try_cast(amount AS DOUBLE) IS NOT NULL"))
)

print("Missing-order payments:", missing_order_payments.count())

In [0]:
(
    missing_order_payments
    .withColumn("rejection_reason", lit("ORDER_NOT_FOUND_IN_SILVER"))
    .write
    .format("delta")
    .mode("append")
    .saveAsTable(f"{catalog}.{schema_name}.quarantine_payments")
)

In [0]:
spark.table(
    f"{catalog}.{schema_name}.quarantine_payments"
).groupBy("rejection_reason").count().show()